## Windows setup notes

If you are running this notebook on Windows, the main thing that may need changing is the FFmpeg and FFprobe path.

The notebook may currently use Mac-specific paths like:

`ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg"`

`ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"`

If FFmpeg is installed and available in Command Prompt or PowerShell, change the paths to:

`ffmpeg_path="ffmpeg"`

`ffprobe_path="ffprobe"`

If FFmpeg is not on PATH, use the full path to the .exe files, for example:

`ffmpeg_path=r"C:\ffmpeg\bin\ffmpeg.exe"`

`ffprobe_path=r"C:\ffmpeg\bin\ffprobe.exe"`

To confirm FFmpeg works on Windows, open Command Prompt and run:

`ffmpeg -version`

`ffprobe -version`

If those commands work, you can usually just use:

`ffmpeg_path="ffmpeg"`

`ffprobe_path="ffprobe"`

In [ ]:
from pathlib import Path
import subprocess
import json


def get_video_duration(video_path, ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"):
    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    cmd = [
        ffprobe_path,
        "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_file)
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("ffprobe failed while reading video duration")

    data = json.loads(result.stdout)
    return float(data["format"]["duration"])


def escape_drawtext(text: str) -> str:
    return (
        text.replace("\\", r"\\\\")
            .replace(":", r"\:")
            .replace("'", r"\'")
            .replace(",", r"\,")
            .replace("[", r"\[")
            .replace("]", r"\]")
            .replace("%", r"\%")
            .replace("(", r"\(")
            .replace(")", r"\)")
    )


def make_6_video_compilation(
    video_paths,
    section_titles,
    output_path="my_2min_compilation.mp4",
    target_total_seconds=120,
    width=1280,
    height=720,
    fps=30,
    ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg",
    ffprobe_path="/opt/homebrew/opt/ffmpeg-full/bin/ffprobe"
):
    if len(video_paths) != 6:
        raise ValueError("You must provide exactly 6 video paths.")

    if len(section_titles) != 6:
        raise ValueError("You must provide exactly 6 section titles.")

    for path in video_paths:
        if not Path(path).exists():
            raise FileNotFoundError(f"video file not found: {path}")

    target_each = target_total_seconds / 6
    durations = [get_video_duration(path, ffprobe_path=ffprobe_path) for path in video_paths]

    cmd = [ffmpeg_path, "-y"]
    for path in video_paths:
        cmd += ["-i", str(path)]

    filter_parts = []

    for i, (duration, title) in enumerate(zip(durations, section_titles)):
        setpts_factor = target_each / duration
        safe_title = escape_drawtext(title)

        part = (
            f"[{i}:v]"
            f"setpts={setpts_factor}*(PTS-STARTPTS),"
            f"trim=duration={target_each},"
            f"fps={fps},"
            f"scale={width}:{height}:force_original_aspect_ratio=decrease,"
            f"pad={width}:{height}:(ow-iw)/2:(oh-ih)/2,"
            f"setsar=1,"
            f"format=yuv420p,"
            f"drawtext="
            f"text='{safe_title}':"
            f"x=(w-text_w)/2:"
            f"y=40:"
            f"fontsize=36:"
            f"fontcolor=white:"
            f"box=1:"
            f"boxcolor=black@0.6:"
            f"boxborderw=12"
            f"[v{i}]"
        )   
        filter_parts.append(part)

    concat_inputs = "".join([f"[v{i}]" for i in range(6)])
    filter_parts.append(f"{concat_inputs}concat=n=6:v=1:a=0[outv]")

    filter_complex = ";".join(filter_parts)

    cmd += [
        "-filter_complex", filter_complex,
        "-map", "[outv]",
        "-an",
        "-c:v", "libx264",
        "-preset", "medium",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        output_path
    ]

    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- FFmpeg stderr ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {result.returncode}")

    print(f"\nDone. Output saved to: {output_path}")
    return output_path

In [ ]:
video_paths = [
    "video1.mp4",
    "video2.mp4",
    "video3.mp4",
    "video4.mp4",
    "video5.mp4",
    "video6.mp4"
]

# keep titles short or function may fail
section_titles = [
    "SVM",           # video1
    "Random Forest", # video2
    "Transformer",   # video3
    "CNN",           # video4
    "DeepLabV3",     # video5
    "U-Net"          # video6
]

output_video = make_6_video_compilation(
    video_paths=video_paths,
    section_titles=section_titles,
    output_path="my_2min_compilation.mp4"
)

output_video

## Trim to 2 minutes


In [ ]:
from pathlib import Path
import subprocess

def trim_video_to_2_minutes_and_delete_original(
    video_path,
    output_path=None,
    ffmpeg_path="/opt/homebrew/opt/ffmpeg-full/bin/ffmpeg",
    target_seconds=120
):
    video_file = Path(video_path)

    if not video_file.exists():
        raise FileNotFoundError(f"video file not found: {video_file}")

    if output_path is None:
        output_path = str(video_file.with_name(video_file.stem + "_trimmed_2min.mp4"))

    output_file = Path(output_path)

    cmd = [
        ffmpeg_path,
        "-y",
        "-i", str(video_file),
        "-t", str(target_seconds),
        "-c:v", "libx264",
        "-c:a", "aac",
        "-movflags", "+faststart",
        str(output_file)
    ]

    print("running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("\n--- ffmpeg stderr ---")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed with exit code {result.returncode}")

    if not output_file.exists():
        raise RuntimeError("trimmed output file was not created")

    video_file.unlink()
    print(f"\ndeleted original: {video_file}")
    print(f"done. output saved to: {output_file}")

    return str(output_file)

In [ ]:
trimmed_video = trim_video_to_2_minutes_and_delete_original(
    "my_2min_compilation.mp4"
)

trimmed_video